  # Mount GDrive

In [ ]:
#Your google drive is made accesible to Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive/')
    %cd /content/drive/MyDrive/TFGs_TFM_2023-2024/TFGs_propuestas_Física/Guillermo_Gracia_Rebullida/tfg_guillermo/code/
    %ls -lht
    # To import own packages set local path in packages syspath
    import sys
    sys.path.insert(0,"./")
except ImportError:
    print("You are not in google.colab!!")
    import os
    # List files in the current working directory
    files_in_directory = os.listdir()
    for file in files_in_directory:
        print(file)

Mounted at /content/drive/
/content/drive/MyDrive/TFGs_TFM_2023-2024/TFGs_propuestas_Física/Guillermo_Gracia_Rebullida/tfg_guillermo/code
total 77K
-rw------- 1 root root  54K Mar 11 16:05 'Transporte lineal (model.fit).py'
-rw------- 1 root root  14K Mar 11 15:59 'Transporte lineal (sin model.fit).ipynb'
-rw------- 1 root root 4.2K Feb 15 10:57 'Transporte lineal (sin model.fit).py'
-rw------- 1 root root 4.8K Feb 15 10:57 'Transporte lineal (model.fit) (1).py'


# Main imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense
import imageio.v2 as imageio

# PINN and initial conditions

In [ ]:
c=tf.convert_to_tensor(1.0,dtype=tf.float64)

def seno(x):
    return tf.sin(x)

tf.keras.utils.get_custom_objects()['seno'] = seno

def u0(x):
    #return tf.sin(2*np.pi*x)
    return x

#parametros redes
nInput=2
nOutput=1
nhiddenlayers=1
nnhiddenlayers=100
n_train=100
n_train0=100
n_border=100

#hiperparametros
lr=0.01
epochs=100
batch_size=1


#parametros de entrenamiento
xmin=-1
xmax=1
tmin=0
tmax=1
N_STEPS=5000
log_each=500

#nn

model=Sequential()
model.add(Input(shape=(nInput,)))
model.add(Dense(nnhiddenlayers,activation='seno'))
model.add(Dense(1,activation=None))

# Metrica, perdida y optimizador
loss = tf.keras.losses.MeanSquaredError()
metrics = tf.keras.metrics.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(lr)

model.compile(loss=loss,optimizer=optimizer,metrics=[metrics])

zeros = tf.zeros(n_border,dtype=tf.float64)
ones = tf.ones(n_border,dtype=tf.float64)

for step in range(0,N_STEPS+1):


    x_train=np.random.uniform(xmin,xmax,size=n_train)
    t_train=np.random.uniform(tmin,tmax,size=n_train)
    tensor_train=np.column_stack((x_train,t_train))
    tensor_train=tf.convert_to_tensor(tensor_train,dtype=tf.float64)

    x_train0=np.random.uniform(xmin,xmax,size=n_train0)
    t_train0=np.zeros(n_train0)
    tensor_train0=np.column_stack((x_train0,t_train0))
    tensor_train0=tf.convert_to_tensor(tensor_train0,dtype=tf.float64)

    t = tf.convert_to_tensor(np.random.uniform(tmin,tmax,n_border),dtype=tf.float64)
    tensor_border1 = tf.stack([zeros,t], axis=-1)
    tensor_border2 = tf.stack([ones,t], axis=-1)


    with tf.GradientTape(persistent=True) as tape:
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch(tensor_train)
            tape2.watch(tensor_train0)
            uNN = model(tensor_train,training=True)
            uNN0 = model(tensor_train0,training=True)

            u00 = u0(x_train0)
            u00=tf.expand_dims(u00,axis=1)

            uNNborder1 = model(tensor_border1,training=True)
            uNNborder2 = model(tensor_border2,training=True)

        partial_derivatives = tape2.gradient(uNN,tensor_train)
        partial_loss = model.compiled_loss(partial_derivatives[:,1], -c*partial_derivatives[:,0])
        zero_loss = model.compiled_loss(uNN0, u00)
        border_loss = model.compiled_loss(uNNborder1,uNNborder2)
        loss=partial_loss+zero_loss+tf.cast(border_loss, dtype=tf.float64)

    gradients=tape.gradient(loss,model.trainable_weights)
    model.optimizer.apply_gradients(zip(gradients, model.trainable_weights))
    model.compiled_metrics.update_state(u0(x_train-c*t_train), uNN)

    if step % log_each == 0:
        print(f'{step}/{N_STEPS} partial_loss {partial_loss:.5f} zero_loss {zero_loss:.5f}  border_loss {border_loss:.5f}')
        print(metrics(u0(x_train-c*t_train),uNN))

0/5000 partial_loss 0.14366 zero_loss 0.56943  border_loss 0.11733
tf.Tensor(0.78115064, shape=(), dtype=float32)
500/5000 partial_loss 0.03498 zero_loss 0.11453  border_loss 0.04658
tf.Tensor(0.32615328, shape=(), dtype=float32)
1000/5000 partial_loss 0.04736 zero_loss 0.13559  border_loss 0.03387
tf.Tensor(0.29435828, shape=(), dtype=float32)


KeyboardInterrupt: 

# Time evolution (gif)

In [ ]:
t_values = np.linspace(0,1, 50)  # Valores de tiempo en los que quieres evaluar la función

# Crea un bucle para cada valor de tiempo
images = []
for t in t_values:
    # Define la malla de puntos (x, t)
    x = np.linspace(xmin, xmax, 50)
    X, T = np.meshgrid(x, [t])

    # Evalúa la función real y la salida de la red neuronal en la malla de puntos
    Z_real = u0(x-c*t)
    t_array=np.zeros(50)
    for i in range (0,50):
        t_array[i]=t
    tensor=np.column_stack((x,t_array))

    Z_nn=model(tensor)

    # Grafica las funciones
    plt.figure(figsize=(8, 6))
    plt.plot(x, Z_real, label='Real')
    plt.plot(x, Z_nn, label='Neural Network')
    plt.xlabel('x')
    plt.ylabel('Function value')
    plt.title(f'Comparison at t={t}')
    plt.legend()

    # Guarda la imagen como un archivo temporal
    filename = f'comparison_t_{t:.2f}.png'
    plt.savefig(filename)
    images.append(imageio.imread(filename))
    plt.close()

# Combina las imágenes en un GIF
imageio.mimsave('comparison.gif', images)

In [ ]:
plt.plot(x,x)